In [13]:
from openai import OpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
import os
from dotenv import load_dotenv

# Load variables from .env file into os.environ
load_dotenv()

# Read the endpoint from the environment
endpoint = os.getenv("OPENAI_ENDPOINT")
if not endpoint:
    raise ValueError("OPENAI_ENDPOINT environment variable is missing or empty.")

# Azure OpenAI
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(), "https://ai.azure.com/.default"
)
openai_client = OpenAI(  
  base_url = endpoint,  
  api_key=token_provider,
)

# Generate a response using the OpenAI-compatible client

In [11]:
# SIMPLE
response = openai_client.responses.create(
    model="gpt-4.1-nano",  # Your model deployment name
    input="What is Microsoft Foundry?"
)

In [2]:
# DETAILED
response = openai_client.responses.create(
    model="gpt-4.1-nano",
    instructions="You are a helpful AI assistant that answers questions clearly and concisely.",
    input="Write a creative story about AI.",
    temperature=0.8,  # Higher temperature for more creativity
    max_output_tokens=200  # Limit response length
)

In [12]:
# Display the response
print(response.output_text)

# Understanding response structure
print(f"Response: {response.output_text}")
print(f"Response ID: {response.id}")
print(f"Tokens used: {response.usage.total_tokens}")
print(f"Status: {response.status}")

Microsoft Foundry is a platform designed by Microsoft to support manufacturing and industrial companies in their digital transformation efforts. It provides tools and solutions for creating, managing, and deploying industrial applications, often emphasizing integration with Microsoft’s cloud services like Azure.

Key features of Microsoft Foundry include:

- **Industrial Application Development:** Enables creation of custom applications tailored to manufacturing processes.
- **Data Integration:** Connects various industrial devices, sensors, and existing enterprise systems.
- **Cloud-Enabled Solutions:** Leverages Azure for scalable, secure data storage, analytics, machine learning, and IoT capabilities.
- **Collaboration and Workflow Management:** Facilitates collaboration across teams and streamlines operational workflows.
- **Security and Compliance:** Ensures data security aligned with industry standards.

It's part of Microsoft’s broader initiative to empower industries to harness

# Working with Foundry direct models
When using the FoundrySDK or AzureOpenAI client to connect to a project endpoint, the Responses API works with both Azure OpenAI models and Foundry direct models (such as Microsoft Phi, DeepSeek, or other models hosted directly in Microsoft Foundry):

In [ ]:
# Using a Foundry direct model
response = openai_client.responses.create(
    model="microsoft-phi-4",  # Example Foundry direct model
    instructions="You are a helpful AI assistant that answers questions clearly and concisely.",
    input="What are the benefits of small language models?"
)

print(response.output_text)

# Creating conversational experiences
For more complex conversational scenarios, you can provide system instructions and build multi-turn conversations:

In [4]:
try:
    # Start with initial message
    conversation_history = [
        {
            "type": "message",
            "role": "user",
            "content": "What is machine learning?"
        }
    ]

    # First response
    response1 = openai_client.responses.create(
        model="gpt-4.1-nano",
        input=conversation_history
    )

    print("Assistant:", response1.output_text)

    # Add assistant response to history
    conversation_history += response1.output

    # Add new user message
    conversation_history.append({
        "type": "message",
        "role": "user", 
        "content": "Can you give me an example?"
    })

    # Second response with full history
    response2 = openai_client.responses.create(
        model="gpt-4.1-nano",
        input=conversation_history
    )

    print("Assistant:", response2.output_text)

except Exception as ex:
    print(f"Error: {ex}")

Assistant: Machine learning is a subset of artificial intelligence (AI) that focuses on developing systems that can learn and improve their performance on a task through experience, without being explicitly programmed for every specific case. Instead of following hard-coded rules, machine learning models identify patterns and relationships in data to make predictions or decisions.

In essence, machine learning involves feeding large amounts of data into algorithms, allowing the system to recognize patterns and make informed choices based on new, unseen data. It has applications across many fields, including image recognition, natural language processing, speech recognition, recommendation systems, and more.
Assistant: Certainly! Here's a simple example of machine learning:

**Email Spam Detection**

Imagine you want to build a system that automatically identifies whether an incoming email is spam or not. You can use machine learning for this task.

**How it works:**
1. **Data collectio

## Context window considerations
The previous_response_id parameter links responses together, maintaining conversation context across multiple API calls.

It's important to note that keeping conversation history can increase token usage. For a single run, the active context window can include:
* System instructions (instructions, safety rules)
* Your current prompt
* Conversation history (previous user + assistant messages)
* Tool schemas (functions, OpenAPI specs, MCP tools, etc.)
* Tool outputs (search results, code interpreter output, files)
* Retrieved memory or documents (from memory stores, RAG, file search)

All of these are concatenated, tokenized, and sent to the model together on every request. The SDK helps you manage state, but it doesn't automatically make token usage cheaper.

# Streaming responses

In [7]:
stream = openai_client.responses.create(
    model="gpt-4.1-nano",
    input="Write a short story about a robot learning to paint.",
    stream=True
)
for event in stream:
                if event.type == "response.output_text.delta":
                    print(event.delta, end="")
                elif event.type == "response.completed":
                    response_id = event.response.id

Once upon a time in a bustling city, there was a robot named Arty. Built for precision and logic, Arty’s creators decided to give him a new purpose — to learn how to paint.

At first, Arty studied books and videos about art, analyzing brush strokes, blending colors, and the emotions behind paintings. But when he tried to imitate what he saw, his lines were stiff, and his colors lacked warmth.

One day, Arty’s creator, Lily, handed him a blank canvas and a set of paints. "Try your heart," she said softly.

Arty hesitated, then dipped his brush into vibrant reds and blues. As he painted, something fascinating happened: his mechanical movements grew more fluid, and his colors started to blend in new, unexpected ways. He was beginning to experiment, not just replicate.

Over days, Arty’s paintings became more expressive. His brushstrokes flowed with curiosity, and his colors told stories of moments he’d learned from Lily — the glow of dawn, the gentle sway of trees, the joyful chaos of a c

# Async usage
For high-performance applications, you can use an asynchronous client that allows you to make non-blocking API calls. The real benefit of AsyncOpenAI appears when you run multiple requests concurrently: while one call is waiting on the network, your application can issue other calls or perform other async work rather than blocking. To use it, import AsyncOpenAI instead of OpenAI and use await with each API call:

In [15]:
import asyncio
from openai import AsyncOpenAI

client = AsyncOpenAI(
    base_url=endpoint,
    api_key=token_provider(),  # Notice the () call - passes the actual string, not the callable
)

async def ask(prompt):
    response = await client.responses.create(
        model="gpt-4.1-nano",
        input=prompt
    )
    return response.output_text

async def main():
    # Issue three requests concurrently instead of one after another
    prompts = [
        "Explain quantum computing briefly.",
        "Summarize the benefits of async I/O.",
        "Give a one-sentence definition of machine learning."
    ]
    results = await asyncio.gather(*(ask(p) for p in prompts))
    for prompt, output in zip(prompts, results):
        print(f"Q: {prompt}\nA: {output}\n")

# If no loop is active (plain Python script / CLI), use asyncio.run()
#asyncio.run(main())

# If a loop is already running (e.g., Jupyter / IPython), await it directly
await main()

Q: Explain quantum computing briefly.
A: Quantum computing is a type of computing that leverages the principles of quantum mechanics to process information. Unlike classical computers that use bits (0s and 1s), quantum computers use quantum bits or qubits, which can exist in multiple states simultaneously thanks to phenomena like superposition. Additionally, qubits can become entangled, meaning the state of one qubit can instantly influence another, regardless of distance. These properties allow quantum computers to perform certain complex calculations much faster than classical computers, especially in fields like cryptography, optimization, and simulating quantum systems.

Q: Summarize the benefits of async I/O.
A: Asynchronous I/O (async I/O) offers several advantages, including:

1. **Improved Performance and Scalability:** Async I/O allows programs to handle multiple I/O operations concurrently without blocking the executing thread, leading to efficient utilization of resources an